# Experiment Design

**Philosophy:** Pattern-first, interview-oriented. Each section answers a question an interviewer would ask. Assumes familiarity with basic statistics.

---

## Decision Table

| If you need to... | Go to |
| :--- | :--- |
| Walk through designing a test end-to-end | §1 — Design Checklist |
| Choose what to measure | §2 — Metric Selection |
| Decide what to randomize | §3 — Unit of Randomization |
| Calculate how long to run the test | §4 — Power Analysis & Sample Size |
| Verify the experiment is set up correctly | §5 — Randomization Checks |

---
## §1 — The Design Checklist

When an interviewer says *"walk me through how you'd design an experiment"*, this is the sequence they expect.

```
1. Define the question        What decision will this experiment inform?
2. Identify the unit          What gets randomized? (user, session, page, cluster)
3. Choose metrics             Primary metric + guardrail metrics
4. Determine sample size      Power analysis → minimum detectable effect → duration
5. Design randomization       Simple, stratified, or cluster?
6. Set the stopping rule      Fixed horizon or sequential? Decide BEFORE launching.
7. Run the experiment         Full weeks, no peeking
8. Analyze                    SRM check → balance check → ATE + CI → segments
9. Recommend                  Effect size + confidence + business impact → decision
```

**Interviewer follow-ups to prepare for:**

| Question | Key point in your answer |
| :--- | :--- |
| How long would you run it? | Power analysis — not "until significance" |
| What if the primary metric doesn't move? | Distinguish no effect vs underpowered |
| What if two metrics move in opposite directions? | Guardrail triggered — do not ship |
| How do you handle novelty effect? | Run for multiple weeks, check if effect decays |
| What if users in both groups interact? | Network effect — switch to cluster randomization |

---
## §2 — Metric Selection

Choosing the wrong metric is one of the most common interview failure points. The metric must be **sensitive** (moves when the treatment works), **attributable** (caused by the treatment, not confounders), and **not gameable** (can't be optimized without delivering real value).

### The Metric Hierarchy

```
North Star Metric
│  Long-run business goal — moves slowly, hard to optimize directly
│  Example: annual revenue, DAU, LTV
│
├── Primary Metric (OEC — Overall Evaluation Criterion)
│   What this experiment is designed to move
│   Must be: sensitive enough to detect effect in the experiment window
│   Example: 7-day retention, checkout conversion rate, session length
│
├── Guardrail Metrics
│   Must NOT move (or must not move negatively)
│   Protect against shipping something that wins locally but hurts globally
│   Example: latency, crash rate, revenue per user, unsubscribe rate
│
└── Diagnostic / Debug Metrics
    Help explain WHY the primary metric moved (or didn't)
    Not used for the ship/no-ship decision
    Example: click-through rate, funnel step completion rates
```

### Good vs Bad Primary Metric

| Property | Good | Bad |
| :--- | :--- | :--- |
| Sensitivity | Moves within experiment window | Takes months to show signal (LTV) |
| Attributability | Directly caused by the feature | Influenced by many external factors |
| Not gameable | Hard to inflate without real value | Clicks, impressions (easy to fake) |
| Directional | Clearly higher = better | Ambiguous direction |

### Common Metric Mistakes

- **Vanity metrics** — total page views looks good but doesn't reflect value; use unique users or session depth instead
- **Proxy too far from the North Star** — optimizing click rate may hurt purchase rate downstream
- **Ratio metrics with unequal denominators** — average revenue per user can move just because user composition shifts; check both numerator and denominator
- **Testing too many primary metrics** — multiple testing inflates false positives; pick one primary metric and pre-register it

**Common mistakes:**
- Treating guardrail metrics as secondary — if a guardrail metric degrades, the experiment fails regardless of the primary metric result
- Choosing a metric that's too slow to move (LTV, annual retention) — experiment will be underpowered in any reasonable timeframe; use a leading indicator instead
- Forgetting to specify the direction of the primary metric — "engagement" can mean more or less depending on context

---
## §3 — Unit of Randomization

The unit of randomization is what gets assigned to treatment or control. Choosing the wrong unit causes **false positives** (session-level when user-level is correct) or **infeasibility** (user-level when the treatment affects the whole platform).

### Units and When to Use Each

| Unit | Use when | Risk |
| :--- | :--- | :--- |
| **User** | Feature is user-specific, most common default | Users interact with each other → spillover |
| **Session** | Feature only affects a single visit | Same user can be in both groups → contamination |
| **Page / Request** | Server-side rendering change, no memory needed | Very high contamination risk |
| **Cluster** (geo, device, cohort) | Network effects, social features, marketplace | Lower statistical power, complex analysis |
| **Time** (switchback) | Marketplace / supply-demand coupling | Carryover effects between periods |

### The Contamination Problem

**Session-level randomization when the unit should be user:**
- User A is assigned to treatment in session 1 and control in session 2
- The treatment effect carries over across sessions (learning, muscle memory)
- Sessions are not independent → standard errors are underestimated → false positives
- Fix: randomize at the user level

### Network Effects → Cluster Randomization

When treated users affect control users (social features, marketplaces, shared resources):
- Assign entire clusters (cities, friend groups, cohorts) to treatment or control
- Clusters must be relatively isolated from each other
- Statistical power drops — need more clusters, not just more users
- Analysis: cluster-robust standard errors or mixed-effects models

### Stratified Randomization

When you want to ensure balance on key covariates (platform, country, user tenure):
- Randomize within strata — each stratum gets exactly 50/50 split
- Reduces variance → increases power (similar to blocking in clinical trials)
- Must analyze with stratification weights to get unbiased estimates

**Common mistakes:**
- Randomizing at the session level for features that affect user state — creates within-user contamination and inflated false positive rate
- Using geo-level clusters that are not actually isolated — a user in NYC may interact with users in Boston, making clusters leaky
- Forgetting that cluster randomization requires cluster-robust standard errors — naive t-test underestimates variance

---
## §4 — Power Analysis & Sample Size

Sample size must be determined **before** the experiment runs. Running until you see significance is peeking — it inflates the false positive rate.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# ── The four inputs — fix three, solve for one ───────────────────────────────
# alpha  (α) — false positive rate — standard: 0.05
# power  (1-β) — true positive rate — standard: 0.80
# effect_size (MDE) — minimum effect worth detecting
# n — sample size per group

def sample_size_per_group(baseline, mde_relative, alpha=0.05, power=0.80):
    """
    Compute required sample size per group for a two-sample z-test on proportions.
    baseline     : baseline conversion rate (e.g. 0.10 for 10%)
    mde_relative : minimum detectable effect as relative lift (e.g. 0.05 for +5%)
    """
    p1 = baseline
    p2 = baseline * (1 + mde_relative)          # expected treatment rate
    delta = p2 - p1                              # absolute effect

    z_alpha = stats.norm.ppf(1 - alpha / 2)      # two-sided
    z_beta  = stats.norm.ppf(power)

    pooled_p = (p1 + p2) / 2
    n = (z_alpha * np.sqrt(2 * pooled_p * (1 - pooled_p)) +
         z_beta  * np.sqrt(p1 * (1-p1) + p2 * (1-p2)))**2 / delta**2
    return int(np.ceil(n))

# Example: checkout page redesign
baseline     = 0.10   # 10% conversion rate
mde_relative = 0.05   # want to detect +5% relative lift (10% → 10.5%)
daily_traffic = 5000  # users per day exposed to the test

n = sample_size_per_group(baseline, mde_relative)
days = int(np.ceil(2 * n / daily_traffic))       # both groups combined

print(f"Baseline conversion:    {baseline:.1%}")
print(f"Target conversion:      {baseline*(1+mde_relative):.2%}")
print(f"Required n per group:   {n:,}")
print(f"Total users needed:     {2*n:,}")
print(f"Duration at {daily_traffic:,}/day:  {days} days ({days/7:.1f} weeks)")

# ── MDE curve — how does sample size scale with MDE? ────────────────────────
mde_values = np.linspace(0.01, 0.20, 50)
n_values   = [sample_size_per_group(baseline, m) for m in mde_values]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0b0e14')
for ax in axes:
    ax.set_facecolor('#13171f')
    ax.tick_params(colors='#6b7a99')
    ax.spines[:].set_color('#1e2433')

# Sample size vs MDE
axes[0].plot(mde_values * 100, n_values, color='#4fc4cf', linewidth=2)
axes[0].axvline(mde_relative * 100, color='#f0c674', linestyle='--', linewidth=1.5,
                label=f'Target MDE = {mde_relative:.0%}')
axes[0].axhline(n, color='#7ec98a', linestyle='--', linewidth=1.5,
                label=f'n = {n:,}')
axes[0].set_xlabel('MDE (relative %)', color='#d4dbe8')
axes[0].set_ylabel('Sample size per group', color='#d4dbe8')
axes[0].set_title('Sample Size vs MDE', color='#d4dbe8', fontsize=11)
axes[0].legend(fontsize=9, labelcolor='#d4dbe8', facecolor='#1e2433', edgecolor='#1e2433')
axes[0].grid(True, alpha=0.2, color='#1e2433')

# Power curve — power vs sample size at fixed MDE
n_range  = np.arange(500, 20000, 200)
p1, p2   = baseline, baseline * (1 + mde_relative)
delta    = p2 - p1
se_vals  = np.sqrt(p1*(1-p1)/n_range + p2*(1-p2)/n_range)
z_vals   = delta / se_vals - stats.norm.ppf(0.975)
power_vals = stats.norm.cdf(z_vals)

axes[1].plot(n_range, power_vals * 100, color='#4fc4cf', linewidth=2)
axes[1].axhline(80, color='#f0c674', linestyle='--', linewidth=1.5, label='80% power')
axes[1].axvline(n,  color='#7ec98a', linestyle='--', linewidth=1.5, label=f'n = {n:,}')
axes[1].set_xlabel('Sample size per group', color='#d4dbe8')
axes[1].set_ylabel('Power (%)', color='#d4dbe8')
axes[1].set_title('Power Curve', color='#d4dbe8', fontsize=11)
axes[1].legend(fontsize=9, labelcolor='#d4dbe8', facecolor='#1e2433', edgecolor='#1e2433')
axes[1].grid(True, alpha=0.2, color='#1e2433')

plt.tight_layout()
plt.show()

### How to Estimate MDE in Practice

```
1. Pull baseline metric from historical data (last 4 weeks)
2. Ask: what effect is meaningful for this business decision?
   - Below MDE → not worth shipping even if real
   - At or above MDE → worth shipping if statistically significant
3. Check if the required sample size is feasible given traffic
   - If duration > 4–6 weeks → increase MDE or increase traffic allocation
4. Round up to full weeks to avoid day-of-week bias
```

### Rules of Thumb

| Parameter | Standard | When to adjust |
| :--- | :--- | :--- |
| α (false positive rate) | 0.05 | Lower to 0.01 for high-stakes / irreversible decisions |
| Power (1-β) | 0.80 | Raise to 0.90 for high-stakes decisions |
| Traffic allocation | 50/50 | Use <50% treatment if feature is risky |
| Minimum duration | Full weeks | Always run at least 1–2 full weeks regardless of power |

**Common mistakes:**
- Setting MDE to the smallest effect you could imagine detecting, not the smallest effect worth caring about — produces impractically large sample sizes
- Not accounting for the fact that you need traffic in both groups — total users needed is `2n`, not `n`
- Running for a fixed number of days that isn't a multiple of 7 — captures a biased day-of-week mix

---
## §5 — Randomization Checks

Run these **before looking at the primary metric**. If any check fails, stop — the experiment results are invalid.

In [ ]:
import pandas as pd
from scipy import stats

# ── Check 1: Sample Ratio Mismatch (SRM) ─────────────────────────────────────
# Expected: 50/50 split. Actual split deviating significantly = broken randomization.
# Causes: bot traffic filtered differently, logging bugs, redirect issues.

def check_srm(n_control, n_treatment, expected_split=0.5, alpha=0.01):
    n_total   = n_control + n_treatment
    expected  = [n_total * expected_split, n_total * (1 - expected_split)]
    observed  = [n_control, n_treatment]
    chi2, p   = stats.chisquare(observed, expected)
    srm       = p < alpha
    print(f"Control:   {n_control:,}  ({n_control/n_total:.2%})")
    print(f"Treatment: {n_treatment:,}  ({n_treatment/n_total:.2%})")
    print(f"Chi² = {chi2:.2f},  p = {p:.4f}")
    print(f"SRM detected: {'YES — experiment invalid' if srm else 'No'}")
    return srm

check_srm(n_control=10_012, n_treatment=9_487)   # imbalanced — SRM likely
print()
check_srm(n_control=10_021, n_treatment=9_979)   # balanced — OK

# ── Check 2: Covariate Balance (AA Check) ────────────────────────────────────
# Verify treatment and control groups have the same pre-experiment characteristics.
# If covariates are imbalanced, randomization failed or assignment is biased.

def balance_check(df, group_col, covariates):
    ctrl = df[df[group_col] == 'control']
    trt  = df[df[group_col] == 'treatment']
    rows = []
    for col in covariates:
        t_stat, p_val = stats.ttest_ind(ctrl[col].dropna(), trt[col].dropna())
        rows.append({
            'covariate':  col,
            'ctrl_mean':  ctrl[col].mean(),
            'trt_mean':   trt[col].mean(),
            'p_value':    round(p_val, 4),
            'imbalanced': p_val < 0.05
        })
    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    n_imbalanced = result['imbalanced'].sum()
    print(f"\n{n_imbalanced} of {len(covariates)} covariates imbalanced at p<0.05")
    if n_imbalanced > len(covariates) * 0.05:
        print("WARNING: more imbalance than expected by chance — check randomization")
    return result

# ── Check 3: AA Test ──────────────────────────────────────────────────────────
# Run the analysis pipeline on data where there is NO treatment (both groups see control).
# Expected: no significant difference.
# If significant: false positive rate is inflated — analysis pipeline has a bug.
# Note: AA tests should be run on historical data, not live traffic.

np.random.seed(0)
aa_ctrl = np.random.binomial(1, 0.10, 10_000)   # same rate — no treatment
aa_trt  = np.random.binomial(1, 0.10, 10_000)
_, p_aa = stats.ttest_ind(aa_ctrl, aa_trt)
print(f"\nAA test p-value: {p_aa:.4f}")
print(f"AA test result: {'FAIL — check analysis pipeline' if p_aa < 0.05 else 'PASS'}")

### What Each Check Catches

| Check | What it detects | Action if it fails |
| :--- | :--- | :--- |
| SRM | Broken assignment, logging bug, bot filtering | Stop experiment, investigate pipeline |
| Covariate balance | Biased randomization, self-selection | Stop experiment, check assignment mechanism |
| AA test | Inflated false positive rate in analysis | Fix analysis pipeline before running any AB test |

**Common mistakes:**
- Looking at the primary metric before running SRM check — if SRM exists, the result is meaningless
- Setting SRM alpha at 0.05 — use 0.01 or lower; SRM is so consequential that you want to be conservative
- Running the AA test on live traffic — this wastes real experiment traffic; use historical data simulation instead

---
## Decision Guide

```
Designing an experiment?
└── Always follow: question → unit → metrics → sample size → randomization → stopping rule  (§1)

Choosing what to measure?
├── One primary metric (pre-registered)                                                     (§2)
├── 2–3 guardrail metrics (must not degrade)                                                (§2)
└── Diagnostic metrics (explain why, not for ship decision)                                 (§2)

Choosing the unit of randomization?
├── Feature is user-specific, no social interaction  → user-level                           (§3)
├── Users interact with each other                   → cluster-level                        (§3)
├── Marketplace / supply-demand coupling             → geo or time (switchback)             (§3)
└── Want balance on key covariates                   → stratified randomization             (§3)

Calculating sample size?
├── Fix: α=0.05, power=0.80, MDE from business context → solve for n                       (§4)
├── Duration = 2n / daily_traffic, rounded up to full weeks                                 (§4)
└── If duration > 6 weeks → increase MDE or renegotiate traffic allocation                  (§4)

Verifying the experiment is valid?
├── SRM check first — if fails, stop entirely                                               (§5)
├── Covariate balance check — before looking at primary metric                              (§5)
└── AA test — run on historical data before launching the experiment platform               (§5)
```